# Async Requests

When making HTTP requests, a significant amount of time is spent on **overhead**: establishing connections, waiting for the server to respond, transferring data, etc. During all that waiting, your program is idle even though there is plenty of network, CPU, disk, and memory capacity available.

The `grequests` library lets you send multiple HTTP requests **in parallel** (concurrently), so that while one request is waiting for a response, others can already be on their way. This dramatically reduces total wall-clock time when you need to fetch many URLs.

> **Important:** This is about **I/O concurrency**, not CPU parallelism. We are not using more CPU cores -- we are simply avoiding idle waiting time by overlapping network I/O.

In [ ]:
%pip install grequests

## How `grequests` works

The library provides a simple two-step pattern:

1. **Create unsent request objects** using `grequests.get()`, `grequests.post()`, etc. These calls do *not* send the request yet -- they just prepare it.
2. **Send them all concurrently** using `grequests.map(requests)`. This sends every request in the collection at the same time, waits for all responses, and returns a list of `Response` objects (or `None` for failed requests).

```python
import grequests

# Step 1: build a collection of unsent requests
rs = [grequests.get(url) for url in urls]

# Step 2: send them all at once and collect the responses
responses = grequests.map(rs)
```

The call to `grequests.map()` is **blocking** -- it waits until every request has completed (or failed) before returning.

## Demo

Let's see the basic pattern in action. We create unsent requests for a handful of URLs and then fire them all at once:

In [ ]:
import grequests

urls = [
    'http://www.heroku.com',
    'http://python-tablib.org',
    'http://httpbin.org',
    'http://python-requests.org',
    'http://fakedomain/',
    'http://kennethreitz.com'
]

# Create unsent request objects
rs = [grequests.get(u) for u in urls]

# Send them all concurrently
responses = grequests.map(rs)

for url, response in zip(urls, responses):
    status = response.status_code if response else 'FAILED'
    print(f"{url} -> {status}")

Notice how some responses may be `None` -- those are requests that failed (e.g. the fake domain). The rest complete with an HTTP status code.

## Exercise 1: Fetching Pokemon data in parallel

The [PokeAPI](https://pokeapi.co/) provides data about Pokemon. Each Pokemon can be fetched at `https://pokeapi.co/api/v2/pokemon/{id}`.

Create a list of 10 `grequests` GET requests for Pokemon with IDs 1 through 10. Use `grequests.map()` to send them all at once. Then print each response's status code.

**Explanation:** We use a list comprehension to create 10 unsent `grequests.get()` objects, one for each Pokemon ID. Then `grequests.map()` sends them all concurrently and returns a list of response objects. We iterate over the responses and print each status code. Since all URLs are valid PokeAPI endpoints, we expect all status codes to be 200.

In [ ]:
import grequests

urls = [f"https://pokeapi.co/api/v2/pokemon/{i}" for i in range(1, 11)]
rs = [grequests.get(url) for url in urls]
responses = grequests.map(rs)

for url, response in zip(urls, responses):
    print(f"{url} -> {response.status_code}")

## Exercise 2: Measuring the speedup

One of the main benefits of async requests is speed. In this exercise, compare **parallel** requests (using `grequests`) with **sequential** requests (using the regular `requests` library) for fetching 20 Pokemon (IDs 1-20).

Use `time.time()` to measure the duration of each approach. Print both durations and the speedup factor (sequential time / parallel time).

**Explanation:** We time both approaches by recording `time.time()` before and after each block. For the parallel approach, we build all requests with `grequests.get()` and send them with `grequests.map()`. For the sequential approach, we use a simple loop with `requests.get()`. The speedup factor is the ratio of sequential duration to parallel duration. Typically you will see a significant speedup (e.g. 3-10x) because the parallel approach overlaps all the network wait times.

In [ ]:
import grequests
import requests
import time

urls = [f"https://pokeapi.co/api/v2/pokemon/{i}" for i in range(1, 21)]

# Parallel requests using grequests
start_parallel = time.time()
rs = [grequests.get(url) for url in urls]
parallel_responses = grequests.map(rs)
end_parallel = time.time()
parallel_duration = end_parallel - start_parallel

# Sequential requests using requests
start_sequential = time.time()
sequential_responses = [requests.get(url) for url in urls]
end_sequential = time.time()
sequential_duration = end_sequential - start_sequential

# Print both durations and the speedup factor
print(f"Parallel duration:   {parallel_duration:.2f} seconds")
print(f"Sequential duration: {sequential_duration:.2f} seconds")
print(f"Speedup factor:      {sequential_duration / parallel_duration:.2f}x")

## Timeouts and exception handling

When sending many requests in parallel, some may fail or take too long. `grequests` provides two mechanisms to deal with this:

### Timeout

You can pass a `timeout` parameter (in seconds) to each request:

```python
grequests.get('http://example.com', timeout=2)
```

If the server does not respond within the timeout, the request raises an exception.

### Exception handler

By default, failed requests silently return `None`. To handle exceptions explicitly, pass an `exception_handler` callback to `grequests.map()`. This function receives the request object and the exception:

```python
def handler(request, exception):
    print(f"Request to {request.url} failed: {exception}")

grequests.map(requests, exception_handler=handler)
```

## Exercise 3: Timeouts and exception handling

Create a list of `grequests` GET requests for 5 Pokemon URLs (IDs 1-5) with a very short timeout of **0.001 seconds** (so they are almost guaranteed to fail).

Write an exception handler function that prints `"Request failed: {url}"` (where `{url}` is the URL of the failed request). Pass this handler to `grequests.map()`.

**Explanation:** We create requests with an extremely short timeout (0.001 seconds), which is far too short for any real network request to complete. This forces every request to fail with a timeout exception. Our `exception_handler` function receives the failed request object and the exception, and prints the URL that failed. The `request.url` attribute gives us the original URL. All requests will fail and trigger the handler, and the resulting list will contain only `None` values.

In [ ]:
import grequests

urls = [f"https://pokeapi.co/api/v2/pokemon/{i}" for i in range(1, 6)]

def exception_handler(request, exception):
    print(f"Request failed: {request.url}")

rs = [grequests.get(url, timeout=0.001) for url in urls]
responses = grequests.map(rs, exception_handler=exception_handler)

print(f"\nResponses: {responses}")

## Exercise 4: Building a reusable `fetch_all` function

Write a function `fetch_all(urls, timeout=5)` that:

1. Takes a list of URLs and an optional timeout (default 5 seconds).
2. Creates async GET requests for each URL with the given timeout.
3. Handles exceptions by returning `None` for failed requests (i.e., the result list should contain `Response` objects for successful requests and `None` for failed ones).
4. Returns the list of responses.

Test it with a mix of valid and invalid URLs.

**Explanation:** We encapsulate the full `grequests` pattern into a reusable function. The function creates a list of unsent GET requests (each with the specified timeout) and sends them with `grequests.map()`. We define a local exception handler that does nothing (just passes), which means failed requests will simply appear as `None` in the result list. This is the default behavior of `grequests.map()`, but providing an explicit handler prevents any unhandled exception warnings. The function returns the list directly, giving the caller `Response` objects for successes and `None` for failures.

In [ ]:
import grequests

def fetch_all(urls, timeout=5):
    def exception_handler(request, exception):
        pass  # Failed requests will be None in the results

    rs = [grequests.get(url, timeout=timeout) for url in urls]
    return grequests.map(rs, exception_handler=exception_handler)

# Test your function
test_urls = [
    "https://pokeapi.co/api/v2/pokemon/1",
    "https://pokeapi.co/api/v2/pokemon/2",
    "http://fakedomain/",
    "https://pokeapi.co/api/v2/pokemon/3",
]

results = fetch_all(test_urls, timeout=5)
for url, resp in zip(test_urls, results):
    if resp:
        print(f"{url} -> {resp.status_code}")
    else:
        print(f"{url} -> None (failed)")

## Summary

In this notebook you learned:

- **`grequests.get(url)`** creates an unsent request object.
- **`grequests.map(requests)`** sends all requests concurrently and returns a list of responses.
- Parallel I/O can provide a significant **speedup** over sequential requests, especially when fetching many URLs.
- Use the **`timeout`** parameter to limit how long each request waits for a response.
- Use an **`exception_handler`** callback to handle failed requests gracefully.
- These techniques are about **I/O concurrency** (overlapping wait times), not CPU parallelism.

For more details, see the [grequests documentation on GitHub](https://github.com/spyoungtech/grequests).